## Project Description: Next Word Prediction Using LSTM
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

- 1) Data Collection: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.
- 2) Data Preprocessing: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.
- 3) Model Building: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.
- 4) Model Training: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.
- 5) Model Evaluation: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.
- 6) Deployment: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

## Trainable parameters
- All the weights and biases are known as Trainable parameters.

In [1]:

import nltk
nltk.download('gutenberg') # This import has a file called 'shakespeare-hamlet.txt' which is our dataset
from nltk.corpus import gutenberg
import  pandas as pd

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,GRU

import pickle



[nltk_data] Downloading package gutenberg to /Users/ankur/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
2026-02-23 18:06:01.351971: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
#1 ) Load the dataset.
# - Copy the dataset to a different file 
# - From: shakespeare-hamlet.txt
# - To: ./resources/data/hamlet.txt

data=gutenberg.raw('shakespeare-hamlet.txt')
with open('./resources/data/hamlet.txt','w') as file:
    file.write(data.lower())

In [ ]:
#2) Data processing
#2.1) Convert the 'text' in hamlet.txt into numerical vectors using N-Gram technique.
#2.1.1) Read the data from the file 'hamlet.txt'

# tokenizer.fit_on_texts() and tokenizer.texts_to_sequences()
#A) tokenizer.fit_on_texts() LEARNS convert words (from hamlet.txt) to numbers
# - tokenizer.fit_on_texts() has to be called before calling tokenizer.texts_to_sequences()

#B) tokenizer.texts_to_sequences() - Converts text to integer sequences
# - By tokenization we are creating indexes for words 
# - e.g. words = The is
# - e.g. tokens= 10  20
# - These indexes are where we have '1' using OHE for this word. These indexes (10,20) are the numerical representation of the inputText.
with open('./resources/data/hamlet.txt','r') as file:
    hamlet_text=file.read().lower()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([hamlet_text])
total_words = len(tokenizer.word_index) + 1
print(f'Total number of words in the vocabulary: {total_words}')
print('word index at which the give word is 1 (using OHE)',tokenizer.word_index)

Total number of words in the vocabulary: 4818
word index at which the give word is 1 (using OHE) {'the': 1, 'and': 2, 'to': 3, 'of': 4, 'i': 5, 'you': 6, 'a': 7, 'my': 8, 'it': 9, 'in': 10, 'that': 11, 'ham': 12, 'is': 13, 'not': 14, 'his': 15, 'this': 16, 'with': 17, 'your': 18, 'but': 19, 'for': 20, 'me': 21, 'lord': 22, 'as': 23, 'what': 24, 'he': 25, 'be': 26, 'so': 27, 'him': 28, 'haue': 29, 'king': 30, 'will': 31, 'no': 32, 'our': 33, 'we': 34, 'on': 35, 'are': 36, 'if': 37, 'all': 38, 'then': 39, 'shall': 40, 'by': 41, 'thou': 42, 'come': 43, 'or': 44, 'hamlet': 45, 'good': 46, 'do': 47, 'hor': 48, 'her': 49, 'let': 50, 'now': 51, 'thy': 52, 'how': 53, 'more': 54, 'they': 55, 'from': 56, 'enter': 57, 'at': 58, 'was': 59, 'oh': 60, 'like': 61, 'most': 62, 'there': 63, 'well': 64, 'know': 65, 'selfe': 66, 'would': 67, 'them': 68, 'loue': 69, 'may': 70, "'tis": 71, 'vs': 72, 'sir': 73, 'qu': 74, 'which': 75, 'did': 76, 'why': 77, 'laer': 78, 'giue': 79, 'thee': 80, 'ile': 81, 'must

In [ ]:
# Step 2.2) Create vector matrix for each line in hamlet.txt using n-gram technique
# - Origional text = hamlet_text
# - Vector representation (using n-gram) = all_n_gram_sequences

# - line = each line in hamlet.txt e.g. actus primus. scoena prima
# - vector_matrix_for_current_line = vector representation for 'line' e.g. [1180, 1889, 1890, 1891]

# - n_gram_sequences = (all possible n-grams for the current line) e.g.
# - n_gram_sequence: [1180, 1889]
# - n_gram_sequence: [1180, 1889, 1890]
# - n_gram_sequence: [1180, 1889, 1890, 1891]
all_n_gram_sequences=[]

for line in hamlet_text.split('\n'):

    # [line] = array of lines, [line][0] = current line
    vector_matrix_for_current_line=tokenizer.texts_to_sequences([line])[0] 
    
    print(f"text =  {line} , vector_matrix =  {vector_matrix_for_current_line}")
    for i in range(1,len(vector_matrix_for_current_line)):
        # Python slicing syntax: list[start:end] -> start is inclusive and end is exclusive, step is optional
        n_gram_sequence=vector_matrix_for_current_line[:i+1]
        all_n_gram_sequences.append(n_gram_sequence)
        print(f'n_gram_sequence: {n_gram_sequence}')   

print(f'all_n_gram_sequences: {all_n_gram_sequences}')        



text =  [the tragedie of hamlet by william shakespeare 1599] , vector_matrix =  [1, 687, 4, 45, 41, 1886, 1887, 1888]
n_gram_sequence: [1, 687]
n_gram_sequence: [1, 687, 4]
n_gram_sequence: [1, 687, 4, 45]
n_gram_sequence: [1, 687, 4, 45, 41]
n_gram_sequence: [1, 687, 4, 45, 41, 1886]
n_gram_sequence: [1, 687, 4, 45, 41, 1886, 1887]
n_gram_sequence: [1, 687, 4, 45, 41, 1886, 1887, 1888]
text =   , vector_matrix =  []
text =   , vector_matrix =  []
text =  actus primus. scoena prima. , vector_matrix =  [1180, 1889, 1890, 1891]
n_gram_sequence: [1180, 1889]
n_gram_sequence: [1180, 1889, 1890]
n_gram_sequence: [1180, 1889, 1890, 1891]
text =   , vector_matrix =  []
text =  enter barnardo and francisco two centinels. , vector_matrix =  [57, 407, 2, 1181, 177, 1892]
n_gram_sequence: [57, 407]
n_gram_sequence: [57, 407, 2]
n_gram_sequence: [57, 407, 2, 1181]
n_gram_sequence: [57, 407, 2, 1181, 177]
n_gram_sequence: [57, 407, 2, 1181, 177, 1892]
text =   , vector_matrix =  []
text =    barnar

In [5]:
# Step 2.3) Padding the sequences to make them of same length.

# Step 2.3.1) Find the length of the longest n-gram sequence in 'all_n_gram_sequences' 
# - max_length_of_n_gram_sequence = Length of the longest n-gram sequence in 'all_n_gram_sequences' 
# - Pad by this amount all of the n-gram sequences in 'all_n_gram_sequences' to make them of same length.
max_length_of_n_gram_sequence=max([len(sequence) for sequence in all_n_gram_sequences])
print(f'max_length_of_n_gram_sequence: {max_length_of_n_gram_sequence}')

# Step 2.3.2) Pad all of the n-gram sequences in 'all_n_gram_sequences' to make them of same length.
all_n_gram_sequences_after_padding = np.array(pad_sequences(all_n_gram_sequences, maxlen=max_length_of_n_gram_sequence, padding='pre'))
print(f'all_n_gram_sequences after padding: {all_n_gram_sequences_after_padding}')


max_length_of_n_gram_sequence: 14
all_n_gram_sequences after padding: [[   0    0    0 ...    0    1  687]
 [   0    0    0 ...    1  687    4]
 [   0    0    0 ...  687    4   45]
 ...
 [   0    0    0 ...    4   45 1047]
 [   0    0    0 ...   45 1047    4]
 [   0    0    0 ... 1047    4  193]]


In [6]:
# Step 3 - Create the training data (x and y) for the model. 
# Step 3.1) - Split into x and y
# - x = all_n_gram_sequences_after_padding[:,:-1] -> all columns except the last column
# - y = all_n_gram_sequences_after_padding[:,-1] -> only the last column
# - (NOTE it is (x = ':,-1') and (y = :,:-1)
x,y=all_n_gram_sequences_after_padding[:,:-1],all_n_gram_sequences_after_padding[:,-1]
print(f'x: {x}')
print(f'y: {y}')

x: [[   0    0    0 ...    0    0    1]
 [   0    0    0 ...    0    1  687]
 [   0    0    0 ...    1  687    4]
 ...
 [   0    0    0 ...  687    4   45]
 [   0    0    0 ...    4   45 1047]
 [   0    0    0 ...   45 1047    4]]
y: [ 687    4   45 ... 1047    4  193]


In [ ]:

# Step 3.2) Convert 'y' from 1-D array to 2-D array using one-hot encoding (OHE) technique.
# - 'y' Before using to_categorical() function
# - y = 1-D numpy array, shape of y = (25732,), it only has rows, no columns since it is 1-D.
print('shape of y before using to_categorical() function (1-D array)= ',y.shape)

# 'y' After using to_categorical() function
# - to_categorical() turns each integer in y into a one-hot(OHE) vector of length total_words.
# - to_categorical() will also make 'y' from 1-D numpy array to 2-D numpy array, shape of y will become (25732, 4818) 
# e.g.
# - if y = [2], Here '2' is an index number converted from text earlier
# - if num_classes = 5
# - y = to_categorical(y,num_classes=5) = [[0,0,1,0,0]], index of 2 is '1' all others are '0', done using OHE.
# - Each label becomes a vector of length 5(here).


# Output observation
# - 25372 = total number of rows, when it was 1-D array
# - 4818 = total columns (given by total_words = = 4818 from Step 2.1.1)
y=tf.keras.utils.to_categorical(y,num_classes=total_words)

print(f'y after one hot encoding: {y}')
print('shape of y after using to_categorical() function (2-D array) = ',y.shape)

shape of y before using to_categorical() function (1-D array)=  (25732,)
y after one hot encoding: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
shape of y after using to_categorical() function (2-D array) =  (25732, 4818)


In [8]:
# Step 3.3) Train test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [9]:
# Step 4) Create the Model 
# Step 4.1) Create the Model using LSTM
# - max_length_of_n_gram_sequence = 14 (from Step 2.3.1) is the length of the longest n-gram sequence in 'all_n_gram_sequences'

# - Sequential is for adding different layers (e.g. Embedding Layer, Dense Layer, LSTM) to the in a sequence to the neural network one by one
lstm_model=Sequential()

# Embedding layer with 100 neurons
lstm_model.add(Embedding(total_words,100,input_length=max_length_of_n_gram_sequence-1)) 

# 1st LSTM layer with 150 nueruons, return_sequences=True is used when we have more than 1 LSTM layer, it returns the full sequence of outputs for each time step instead of just the last output.
lstm_model.add(LSTM(150,return_sequences=True)) 

# Dropout layer with dropout rate of 0.2, it randomly sets 20% of the input units to 0(i.e discards 20% of the neurons) at each update during training time, which helps prevent overfitting.
# The selection is different every iteration
lstm_model.add(Dropout(0.2))

# 2nd LSTM layer with 100 neurons.
lstm_model.add(LSTM(100))

# Output layer with nuerons = total_words (4818)
# Activation function = softmax, because it is multiclassification, 
# - We will only predict the next ONE word only, but it can be any word from the vocabulary of 4818 words, so it is multiclassification, hence we use softmax activation function.
lstm_model.add(Dense(total_words,activation="softmax"))

# #Compile the model
# - Categorical cross entropy is used as the loss function for multi-class classification problems where the target variable is one-hot encoded. 
# It measures the dissimilarity between the predicted probability distribution and the true distribution (which is represented as a one-hot encoded vector). 
# The goal of training is to minimize this loss, which means making the predicted probabilities as close as possible to the true labels.
lstm_model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
lstm_model.summary()

/Users/ankur/backup/delta/sn/datascience/workspace/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Step 4.2) Create the Model using GRU
# - max_length_of_n_gram_sequence = 14 (from Step 2.3.1) is the length of the longest n-gram sequence in 'all_n_gram_sequences'

# - Sequential is for adding different layers (e.g. Embedding Layer, Dense Layer, LSTM) to the in a sequence to the neural network one by one
gru_model=Sequential()

# Embedding layer with 100 neurons
gru_model.add(Embedding(total_words,100,input_length=max_length_of_n_gram_sequence-1)) 

# 1st GRU layer with 150 nueruons, return_sequences=True is used when we have more than 1 GRU layer, it returns the full sequence of outputs for each time step instead of just the last output.
gru_model.add(GRU(150,return_sequences=True)) 

# Dropout layer with dropout rate of 0.2, it randomly sets 20% of the input units to 0(i.e discards 20% of the neurons) at each update during training time, which helps prevent overfitting.
# The selection is different every iteration
gru_model.add(Dropout(0.2))

# 2nd GRU layer with 100 neurons.
gru_model.add(GRU(100))

# Output layer with nuerons = total_words (4818)
# Activation function = softmax, because it is multiclassification, 
# - We will only predict the next ONE word only, but it can be any word from the vocabulary of 4818 words, so it is multiclassification, hence we use softmax activation function.
gru_model.add(Dense(total_words,activation="softmax"))

# Compile the model
# - Categorical cross entropy is used as the loss function for multi-class classification problems where the target variable is one-hot encoded. 
# It measures the dissimilarity between the predicted probability distribution and the true distribution (which is represented as a one-hot encoded vector). 
# The goal of training is to minimize this loss, which means making the predicted probabilities as close as possible to the true labels.
gru_model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
gru_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Step 5) Train the model (Note - This takes up a lot of time)
# Step 5.1) Train the LSTM model
lstm_model.fit(x_train,y_train,epochs=100,validation_data=(x_test,y_test),verbose=1,callbacks=[EarlyStopping(monitor='val_loss',patience=50)])

Epoch 1/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 30s 41ms/step - accuracy: 0.0337 - loss: 6.9069 - val_accuracy: 0.0324 - val_loss: 6.7097
Epoch 2/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 37s 58ms/step - accuracy: 0.0379 - loss: 6.4746 - val_accuracy: 0.0422 - val_loss: 6.8016
Epoch 3/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 44s 68ms/step - accuracy: 0.0440 - loss: 6.3419 - val_accuracy: 0.0482 - val_loss: 6.8729
Epoch 4/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 41s 64ms/step - accuracy: 0.0509 - loss: 6.2015 - val_accuracy: 0.0468 - val_loss: 6.8696
Epoch 5/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 42s 66ms/step - accuracy: 0.0547 - loss: 6.0709 - val_accuracy: 0.0525 - val_loss: 6.9130
Epoch 6/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 41s 63ms/step - accuracy: 0.0615 - loss: 5.9287 - val_accuracy: 0.0579 - val_loss: 6.9693
Epoch 7/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 50s 78ms/step - accuracy: 0.0694 - loss: 5.7861 - val_accuracy: 0.0635 - val_loss: 7.0022
Epoch 8/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 46s 72ms/step - accuracy: 0.0750 - loss: 5

In [12]:
# Step 5.2) Train the GRU model
gru_model.fit(x_train,y_train,epochs=100,validation_data=(x_test,y_test),verbose=1,callbacks=[EarlyStopping(monitor='val_loss',patience=50)])

Epoch 1/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 32s 44ms/step - accuracy: 0.0315 - loss: 7.0287 - val_accuracy: 0.0324 - val_loss: 6.9150
Epoch 2/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 27s 42ms/step - accuracy: 0.0342 - loss: 6.5924 - val_accuracy: 0.0361 - val_loss: 6.9677
Epoch 3/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 32s 50ms/step - accuracy: 0.0468 - loss: 6.3884 - val_accuracy: 0.0459 - val_loss: 6.8445
Epoch 4/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 33s 52ms/step - accuracy: 0.0575 - loss: 6.1638 - val_accuracy: 0.0641 - val_loss: 6.7873
Epoch 5/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 34s 52ms/step - accuracy: 0.0714 - loss: 5.9104 - val_accuracy: 0.0589 - val_loss: 6.8648
Epoch 6/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 30s 47ms/step - accuracy: 0.0814 - loss: 5.6519 - val_accuracy: 0.0690 - val_loss: 6.9420
Epoch 7/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 33s 50ms/step - accuracy: 0.0942 - loss: 5.3750 - val_accuracy: 0.0711 - val_loss: 7.0748
Epoch 8/100
644/644 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.1066 - loss: 5

In [13]:
# Step 6 - Predit the next word
# Step 6.1) Create a generic word to predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]  # Ensure the sequence length matches max_sequence_len-1
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None


In [14]:
# Step 6.2) Predict the next word using the LSTM model
input_text="say, what is horatio"
print(f"Input text:{input_text}")
max_sequence_len=lstm_model.input_shape[1]+1
next_word=predict_next_word(lstm_model,tokenizer,input_text,max_sequence_len)
print(f"Next Word PRediction:{next_word}")

Input text:say, what is horatio
Next Word PRediction:i


In [15]:
# Step 6.3) Predict the next word using the GRU model
input_text="say, what is horatio"
print(f"Input text:{input_text}")
max_sequence_len=gru_model.input_shape[1]+1
next_word=predict_next_word(gru_model,tokenizer,input_text,max_sequence_len)
print(f"Next Word PRediction:{next_word}")

Input text:say, what is horatio
Next Word PRediction:you


In [16]:
# Step 7 - Save the models(lstm rnn, gru rnn), tokenizer
# Step 7.1) Save the LSTM and GRU models
# - keras is the latest format, h5 is the older format, keras is better than h5.
lstm_model.save("./resources/dist/lstm_model.keras")
gru_model.save("./resources/dist/gru_model.keras")

# Step 7.2) Saving the Tokenizer
# - Cannot save tokenizer this way - not attribute 'save()'
# - tokenizer.save("./resources/dist/tokenizer.json")   
# - protocol=pickle.HIGHEST_PROTOCOL = Use the most recent, most efficient binary serialization format supported by your current Python version.
with open('./resources/dist/tokenizer.pickle','wb') as file:
    pickle.dump(tokenizer,file,protocol=pickle.HIGHEST_PROTOCOL)